In [1]:
!pip install transformers torch scikit-learn pandas pyvi

   ---------------------------------------- 0.0/8.5 MB ? eta -:--:--
   ---------------------------------------- 0.0/8.5 MB ? eta -:--:--
   - -------------------------------------- 0.3/8.5 MB ? eta -:--:--
   -- ------------------------------------- 0.5/8.5 MB 742.1 kB/s eta 0:00:11
   -- ------------------------------------- 0.5/8.5 MB 742.1 kB/s eta 0:00:11
   --- ------------------------------------ 0.8/8.5 MB 802.7 kB/s eta 0:00:10
   ---- ----------------------------------- 1.0/8.5 MB 842.6 kB/s eta 0:00:09
   ------ --------------------------------- 1.3/8.5 MB 862.7 kB/s eta 0:00:09
   ------ --------------------------------- 1.3/8.5 MB 862.7 kB/s eta 0:00:09
   ------ --------------------------------- 1.3/8.5 MB 862.7 kB/s eta 0:00:09
   ------- -------------------------------- 1.6/8.5 MB 723.4 kB/s eta 0:00:10
   ------- -------------------------------- 1.6/8.5 MB 723.4 kB/s eta 0:00:10
   -------- ------------------------------- 1.8/8.5 MB 751.0 kB/s eta 0:00:09
   --------- 

In [3]:
# Nếu thiếu thì chạy (chỉ 1 lần)
# !pip install transformers torch scikit-learn pandas tqdm

import pandas as pd
import numpy as np
import torch

from torch.utils.data import Dataset, DataLoader
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    get_linear_schedule_with_warmup
)

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score, f1_score

from tqdm import tqdm

In [4]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("🚀 Device:", device)
print("GPU:", torch.cuda.get_device_name(0))

🚀 Device: cuda
GPU: NVIDIA GeForce RTX 3050 Laptop GPU


In [5]:
DATA_PATH = "ML/data/vn_fake_news_combined.csv"

df = pd.read_csv(DATA_PATH)
df = df[['text', 'label']]
df.dropna(inplace=True)

print("Tổng số mẫu:", len(df))
df.head()

Tổng số mẫu: 22929


,text,label
0,Tiền vệ Bruno Fernandes đeo băng đội trưởng th...,0
1,SingaporeMelissa Lim 36 ung_thư vú giai_đoạn 2...,0
2,"Bác sĩ Phạm Thái Ngọc Long, Khoa Cấp cứu, Bệnh...",0
3,khí_hậu chủ_yếu hoa kỳ bất_ổn xã_hội bất_công ...,0
4,TP HCMÔng Bá 60 trò_chuyện đột_ngột im_lặng li...,0


In [6]:
X_train, X_test, y_train, y_test = train_test_split(
    df['text'],
    df['label'],
    test_size=0.2,
    random_state=42,
    stratify=df['label']
)

print("Train:", len(X_train))
print("Test :", len(X_test))

Train: 18343
Test : 4586


In [7]:
MODEL_NAME = "vinai/phobert-base"

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    use_fast=False
)

d:\Code\fake-news\.venv\Lib\site-packages\huggingface_hub\file_download.py:130: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\ADMIN\.cache\huggingface\hub\models--vinai--phobert-base. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


In [8]:
class FakeNewsDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=256):
        self.texts = texts.tolist()
        self.labels = labels.tolist()
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            self.texts[idx],
            truncation=True,
            padding='max_length',
            max_length=self.max_len,
            return_tensors="pt"
        )

        return {
            "input_ids": encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0),
            "labels": torch.tensor(self.labels[idx], dtype=torch.long)
        }

In [9]:
BATCH_SIZE = 8  # RTX 3050 an toàn

train_dataset = FakeNewsDataset(X_train, y_train, tokenizer)
test_dataset  = FakeNewsDataset(X_test, y_test, tokenizer)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader  = DataLoader(test_dataset, batch_size=BATCH_SIZE)

In [10]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2
)

model.to(device)

Loading weights: 100%|██████████| 197/197 [00:00<00:00, 497.44it/s, Materializing param=roberta.encoder.layer.11.output.dense.weight]              
RobertaForSequenceClassification LOAD REPORT from: vinai/phobert-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.decoder.weight          | UNEXPECTED | 
roberta.pooler.dense.bias       | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
roberta.pooler.dense.weight     | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.decoder.bias            | UNEXPECTED | 
classifier.dense.bias           | MISSING    | 
classifier.out_proj.weight      | MISSING    | 
classifier.out_proj.bias        | MISSING    | 
classifier.dense.weight         | MISSING    | 

Notes:
- UNEX

RobertaForSequenceClassification(
  (roberta): RobertaModel(
    (embeddings): RobertaEmbeddings(
      (word_embeddings): Embedding(64001, 768, padding_idx=1)
      (token_type_embeddings): Embedding(1, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
      (position_embeddings): Embedding(258, 768, padding_idx=1)
    )
    (encoder): RobertaEncoder(
      (layer): ModuleList(
        (0-11): 12 x RobertaLayer(
          (attention): RobertaAttention(
            (self): RobertaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): RobertaSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
             

In [11]:
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=2e-5
)

EPOCHS = 3

total_steps = len(train_loader) * EPOCHS

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=int(0.1 * total_steps),
    num_training_steps=total_steps
)

In [12]:
def train_one_epoch(model, loader):
    model.train()
    total_loss = 0

    for batch in tqdm(loader, desc="Training"):
        optimizer.zero_grad()

        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels
        )

        loss = outputs.loss
        loss.backward()

        optimizer.step()
        scheduler.step()

        total_loss += loss.item()

    return total_loss / len(loader)

In [13]:
def evaluate(model, loader):
    model.eval()

    preds = []
    true_labels = []

    with torch.no_grad():
        for batch in tqdm(loader, desc="Evaluating"):
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask
            )

            logits = outputs.logits
            predictions = torch.argmax(logits, dim=1)

            preds.extend(predictions.cpu().numpy())
            true_labels.extend(labels.cpu().numpy())

    return true_labels, preds

In [14]:
for epoch in range(EPOCHS):
    print(f"\n🧠 Epoch {epoch+1}/{EPOCHS}")
    train_loss = train_one_epoch(model, train_loader)
    print(f"Train loss: {train_loss:.4f}")


🧠 Epoch 1/3


Training: 100%|██████████| 2293/2293 [30:36<00:00,  1.25it/s]  


Train loss: 0.2521

🧠 Epoch 2/3


Training: 100%|██████████| 2293/2293 [19:45<00:00,  1.93it/s]


Train loss: 0.1086

🧠 Epoch 3/3


Training: 100%|██████████| 2293/2293 [19:11<00:00,  1.99it/s]

Train loss: 0.0588


In [15]:
y_true, y_pred = evaluate(model, test_loader)

print("Accuracy:", accuracy_score(y_true, y_pred))
print("F1-score:", f1_score(y_true, y_pred))

print("\nClassification Report:")
print(classification_report(y_true, y_pred, digits=4))

Evaluating: 100%|██████████| 574/574 [01:33<00:00,  6.13it/s]

Accuracy: 0.9483209768861753
F1-score: 0.9029086439983613

Classification Report:
              precision    recall  f1-score   support

           0     0.9475    0.9827    0.9648      3304
           1     0.9508    0.8596    0.9029      1282

    accuracy                         0.9483      4586
   macro avg     0.9491    0.9212    0.9338      4586
weighted avg     0.9484    0.9483    0.9475      4586



In [16]:
SAVE_PATH = "ML/models/phobert_fake_news"

model.save_pretrained(SAVE_PATH)
tokenizer.save_pretrained(SAVE_PATH)

print("✅ Đã lưu PhoBERT model tại:", SAVE_PATH)

Writing model shards: 100%|██████████| 1/1 [00:01<00:00,  1.13s/it]

✅ Đã lưu PhoBERT model tại: ML/models/phobert_fake_news
